In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import os

DATA_DIR = "."  # change this to your training_visualization folder path if needed

files = {
    'rejected':   'wandb_export_2026-04-15T21_30_14.105-04_00.csv',
    'margins':    'wandb_export_2026-04-15T21_30_31.360-04_00.csv',
    'chosen':     'wandb_export_2026-04-15T21_30_35.771-04_00.csv',
    'accuracies': 'wandb_export_2026-04-15T21_30_39.939-04_00.csv',
}

dfs = {k: pd.read_csv(os.path.join(DATA_DIR, v)) for k, v in files.items()}

runs = {
    r'$\beta=2.0$': 'dpo-beta2.0_lr5e-06',
    r'$\beta=1.0$': 'dpo-beta1.0_lr5e-06',
    r'$\beta=0.3$ (original)': 'qwen25-7b-dpo-socialiqa',
}

colors = {
    r'$\beta=2.0$':           '#2980b9',
    r'$\beta=1.0$':           '#e74c3c',
    r'$\beta=0.3$ (original)': '#27ae60',
}

def get_series(df, metric_key, run_key):
    step_col = 'train/global_step'
    val_col  = f'{run_key} - train/rewards/{metric_key}'
    sub = df[[step_col, val_col]].dropna()
    return sub[step_col].values, sub[val_col].values

metrics = [
    ('margins',    'Reward margin (chosen - rejected)', 'Reward margin'),
    ('accuracies', 'Reward accuracy',                   'Reward accuracy'),
    ('chosen',     'Chosen reward',                     'Reward (nats)'),
    ('rejected',   'Rejected reward',                   'Reward (nats)'),
]

fig = plt.figure(figsize=(16, 10))
gs  = gridspec.GridSpec(2, 2, hspace=0.38, wspace=0.32)

for idx, (metric_key, title, ylabel) in enumerate(metrics):
    ax = fig.add_subplot(gs[idx // 2, idx % 2])
    df = dfs[metric_key]

    for label, run_key in runs.items():
        try:
            steps, vals = get_series(df, metric_key, run_key)
            ax.plot(steps, vals, label=label, color=colors[label],
                    linewidth=1.5, alpha=0.85)
        except KeyError:
            print(f'Column not found: {run_key} / {metric_key}')

    if metric_key == 'margins':
        ax.axhspan(1.0, 2.0, alpha=0.12, color='green', label='Target range (1-2)')

    if metric_key == 'accuracies':
        ax.axhline(0.90, linestyle='--', color='gray', alpha=0.5,
                   linewidth=1, label='90% threshold')
        ax.set_ylim(0, 1.05)

    ax.set_title(title, fontsize=12)
    ax.set_xlabel('Training step', fontsize=10)
    ax.set_ylabel(ylabel, fontsize=10)
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

plt.suptitle('DPO training dynamics across configurations', fontsize=14, y=1.01)
plt.savefig(os.path.join(DATA_DIR, 'dpo_training_curves.png'),
            dpi=150, bbox_inches='tight')
plt.show()
print('Saved to dpo_training_curves.png')